# HERCULES preparation notebook - read in MakePlanets thermal profiles and write HERCULES yaml



In [1]:
# Record the verions information for these calculation for posterity
import platform
print('python version: ',platform.python_version())
del platform
import matplotlib
print('matplotlib version: ', matplotlib.__version__)
del matplotlib
import numpy
print('numpy version: ', numpy.__version__)
del numpy
import scipy
print('scipy version: ', scipy.__version__)
del scipy

import sys

import woma
import seagen
#print('seagen version: ', seagen.__version__)
#import gadget_sph
from importlib import reload
#reload(gadget)

import swiftsimio as sw
import unyt
from woma.misc import glob_vars as gv
from woma.eos import sesame
from woma.eos.T_rho import T_rho
from woma.misc import utils, io

R_earth = gv.R_earth #6.371e6   # m
M_earth = gv.M_earth #5.9724e24  # kg 
G = gv.G #6.67408e-11  # m^3 kg^-1 s^-2
LEM=3.5E34 #AMof Earth-Moon system in mks

mant_mat_id = 403 #USER INPUT
core_mat_id = 402 #USER INPUT

# First we import python libraries
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from scipy import interpolate
from mpl_toolkits.axes_grid1 import make_axes_locatable
#import colormaps as local_cmaps
import os
from copy import deepcopy
from pathlib import Path
import pkgutil
import h5py
import yaml
print(yaml.__version__)
from HERCULES_structures import *

%run eostable.py

%config InlineBackend.figure_format = 'retina'

plt.rcParams["figure.figsize"] = (20,10) # default figure size in inches

#eospath = os.path.expanduser('~') + '/Work/'
eospath = '../'

python version:  3.9.7
matplotlib version:  3.8.3
numpy version:  1.23.1
scipy version:  1.12.0
6.0.1


In [2]:
# calculate the structure for one planet
class isentrope_class:
    """Class to isentrope data extracted from EOS table."""  # this is a documentation string for this class
    def __init__(self,entropy=None): # self is the default name of the object for internal referencing of the variables in the class
        """A function to initialize the class object.""" # this is a documentation string for this function
        self.entropy = entropy
        self.ND = 0 # number of density points
        self.density     = []   
        self.pressure    = []
        self.temperature = []
        self.soundspeed  = []
        self.energy      = []
        # not going to use all the variables in the file
        self.units = '' # I like to keep a text note in a structure about the units

class planet_profile:
    def __init__(self,file=None):
        self.file=file
        self.M = 0.
        self.cf = 0.
        self.rarr = np.array([])
        self.density = np.array([])
        self.pressure = np.array([])
        self.temperature = np.array([])
        self.energy = np.array([])
        self.entropy = np.array([])
        self.mat = np.array([])
        self.mass = np.array([])
        self.units = 'cgs'
        self.iendcore=0
        if file is not None:
            self.load(file)
        
    def load(self,file='profile.txt'):
        self.rarr,self.density,self.temperature,self.pressure,self.energy,self.entropy,self.mass,self.mat=np.loadtxt(file,skiprows=1,unpack=True)
        self.M=self.mass[-1]
        self.file=file
        
    def convert_to_mks(self):
        if self.units!='mks':
            self.rarr = self.rarr * 1.e-2            #cm to m
            self.density = self.density * 1.e3       #g/cm^3 to kg/m^3
            self.pressure = self.pressure * 1.e-1    #dyne/cm^2 to Pa
            self.energy = self.energy * 1.e-4        #erg/g to J/kg
            self.entropy = self.entropy * 1.e-4      #erg/g/K to J/kg/K
            self.mass = self.mass * 1.e-3            #g to kg
            self.M = self.mass[-1]
            self.units = 'mks'
            
    def convert_to_cgs(self):
        if self.units!='cgs':
            self.rarr = self.rarr * 1.e2            #m to cm
            self.density = self.density * 1.e-3       #kg/m^3 to g/cm^3
            self.pressure = self.pressure * 1.e1    #Pa to dyne/cm^2
            self.energy = self.energy * 1.e4        #J/kg to erg/g
            self.entropy = self.entropy * 1.e4      #J/kg/K to erg/g/K
            self.mass = self.mass * 1.e3            #kg to g
            self.M = self.mass[-1]
            self.units = 'cgs'
            
    def write(self,file='profile.txt'):
        self.convert_to_cgs()
        with open(file,"w") as outfile: # open the file for writing
            outfile.write("#radius[cm] density[g/cm^3] temperature[K] pressure[dyne/cm^2] sp.energy[erg/g] sp.entropy[erg/g/K] menclosed[g] matID\n")
            for i in range(0,len(self.rarr)):
                outfile.write("{:.8e} {:.8e} {:.8e} {:.8e} {:.8e} {:.8e} {:.8e} {:g}\n".format(self.rarr[i],self.density[i],self.temperature[i],self.pressure[i],self.energy[i],self.entropy[i],self.mass[i],self.mat[i]))
        

In [3]:
class HERC_planet:
    def __init__(self,planet=None):
        self.planet=None
        self.M = 0.
        self.Mcore = 0.
        self.Mmant = 0.
        self.Lfinal = 0.
        self.r1 = 0.
        self.rCMB = 0.
        self.basename = ''
        self.mant_profile = ''
        self.core_profile = ''
        if planet is not None:
            self.load(planet)
        
    def load(self,planet):
        if planet.file is None:
            print('Error: tried to load empty planet')
            return
        planet.convert_to_mks()
        self.planet=planet # this is the planet_profile object that defines the HERCULES ICs
        self.M=self.planet.M
        indcore=np.where(self.planet.mat==core_mat_id)
        self.Mcore=self.planet.mass[indcore][-1]
        self.Mmant=self.M-self.Mcore
        self.r1=self.planet.rarr[-1]
        self.rCMB=self.planet.rarr[indcore][-1]
        self.Lfinal=((self.M/M_earth)**2*5+1)*LEM
        # finding basename, expecting the file to be in the planet_profiles folder
        #print(np.char.split(planet.file,'/'))
        split=np.char.split(planet.file,'/').item()[2]
        print(split)
        self.basename=np.char.split(split,'N').item()[0] #assuming the same naming format as in MakePlanets
        
    def write_HERC_profiles(self,wdir='./HERC_profiles/'):
        partstring=['core','mant']
        matids=[core_mat_id,mant_mat_id]
        for i in range(0,2):
            ind=np.where(self.planet.mat==matids[i])
            N=np.size(ind)
            mdenom=self.planet.mass[ind][-1]-i*self.Mcore
            
            fraclist = (self.planet.mass[ind]-i*self.Mcore)/mdenom
            Slist = self.planet.entropy[ind]
            rholist = self.planet.density[ind]
            potlist = -G*self.planet.mass[ind]/(self.planet.rarr[ind]+1)+1.e-10
            ulist = self.planet.energy[ind]
            KElist = np.zeros(N)+1.e-10
            Etotlist = potlist+ulist
            
            fname=wdir+self.basename+'_HERCULES_'+partstring[i]+'_profiles.csv'
            with open(fname,'w') as wfile:
                if i==1:
                    wfile.write('0'+'\n'+wdir+self.basename+''+'\n'+'""'+'\n'+'""'+'\n'+'""'+'\n'+str(0.)+'\n'+str(self.Mmant)+'|'+str(self.Mcore)+'\n'+str(self.r1)+'|'+str(self.rCMB)+'\n')
                else:
                    wfile.write('0'+'\n'+wdir+self.basename+''+'\n'+'""'+'\n'+'""'+'\n'+'""'+'\n'+'""'+'\n'+'""'+'\n'+'""'+'\n')
                    wfile.write('Mass(kg),Entropy(kg(K^-1)(m/s)^2),Densisty(kg/m^3),Potential Energy(kg(m/s)^2),Internal Energy(kg(m/s)^2),Kinetic Energy(kg(m/s)^2),Total Av. Energy(kg(m/s)^2)'+'\n')
                for j in range(N):
                    wfile.write('{:.9e},{:.9e},{:.9e},{:.9e},{:.9e},{:.9e},{:.9e}'.format(fraclist[j],Slist[j],rholist[j],potlist[j],ulist[j],KElist[j],Etotlist[j])+'\n')
            if i==1:
                self.mant_profile=fname
            else:
                self.core_profile=fname
            print('Saved file: ',fname)
            
    def write_yaml(self,wdir='./HERC_yaml/',
                   aspect_init = 0.99,
                   output_dir = './',
                   thermo_var_output = ['T','S','E'],
                   nint_max = 20,
                   toll = 1.e-4,
                   xi_nint_max = 200,
                   xi_toll = 1.e-5,
                   dxi = 1.e-3,
                   omega_param = [1.0,0.0,1.5],
                   material_lay = [100,100],
                   EOS_file_mant = '/home/apostema/hercules/hercules/HERCULES_development/Source_code/Pyrolite_NEW-GADGET-STD-NOTENSION.TXT',
                   EOS_file_core = '/home/apostema/hercules/hercules/HERCULES_development/Source_code/Fe85Si15_NEW-GADGET-STD-NOTENSION.TXT',
                   thermo_profile_mant = None,
                   thermo_profile_core = None,
                   ref_rho = 1.e3,
                   kmax = 6,
                   Nmu = 400,
                   omega_rot = 0.0,
                   Lstart = 0.0,
                   pmin = 1.e6,
                   Lfinal = None,
                   Lstep = None,
                   Lstep_frac = 1.e-2,
                   Ndiv_Lstep = 6,
                   toll_array = 1.e-4,
                   xi_toll_array = 1.e-4,
                   file_suffix = 'input'):
        if thermo_profile_mant is None:
            thermo_profile_mant = self.basename+'_HERCULES_'+'mant'+'_profiles.csv'
        if thermo_profile_core is None:
            thermo_profile_core = self.basename+'_HERCULES_'+'core'+'_profiles.csv'
        if Lfinal is None:
            Lfinal = self.Lfinal
        if Lstep is None:
            Lstep = Lfinal*Lstep_frac
        material_flag_EOS = np.ndarray.tolist(np.full(np.size(material_lay),2))
        flag_thermo_profile = np.ndarray.tolist(np.full(np.size(material_lay),1))
        
        yamldata = dict(
            Main = dict(
                flag_version = 2,
                flag_subversion = [0],
                flag_run_mode = 1),
            Initialization = dict(
                flag_start = 0,
                start_file = None,
                flag_format_start_file = 1,
                aspect_init = aspect_init,
                material_amax_init = [float(self.r1),float(self.rCMB)]),
            Output = dict(
                run_name_base = self.basename,
                flag_naming = 1,
                flag_output_format = 1,
                output_dir = output_dir,
                flag_iter_print = 0,
                thermo_var_output = thermo_var_output),
            Iteration = dict(
                nint_max = nint_max,
                toll = float(toll),
                xi_nint_max = xi_nint_max,
                xi_toll = float(xi_toll),
                dxi = float(dxi),
                flag_Mconc = 2,
                flag_Lconc = 1,
                omega_param = omega_param),
            Planet = dict(
                Nlayers = int(np.sum(material_lay)),
                Nmaterial = int(np.size(material_lay)),
                material_lay = material_lay,
                material_flag_EOS = material_flag_EOS,
                EOS_files_std = [EOS_file_mant,EOS_file_core],
                flag_thermo_profile = flag_thermo_profile,
                thermo_profile_files = [thermo_profile_mant,thermo_profile_mant],
                Mass = [float(self.Mmant),float(self.Mcore)],
                ref_rho = float(ref_rho),
                kmax = kmax,
                Nmu = Nmu,
                omega_rot = float(omega_rot),
                Ltot = float(Lstart),
                pmin = float(pmin)),
            Planet_array_calc = dict(
                Xfinal = float(Lfinal),
                Xstep = float(Lstep),
                Ndiv_Xstep = Ndiv_Lstep,
                flag_array_toll = [0],
                toll_array = float(toll_array),
                xi_toll_array = float(xi_toll_array))
        )
        
        fname=wdir+self.basename+'_input.yml'
        with open(fname, 'w') as wfile:
            yaml.dump(yamldata, wfile, default_style=None, default_flow_style=None, sort_keys=False, width=4096)
        
        print('Saved file: ',fname)
        
#def restart_yaml(sefl,wdir='')

In [4]:
filelist = np.array([])
planetdict = {}
hercdict = {}
for filename in os.listdir("./planet_profiles/"):
    if filename.endswith(".txt") and not filename.endswith(".used_params.txt"): 
        #print(filename)
        filelist = np.append(filelist,filename)
        path = './planet_profiles/'+filename
        planet = planet_profile(file=path)
        planetdict[filename] = planet
        hercplanet = HERC_planet(planet=planet)
        hercdict[filename] = hercplanet
        hercplanet.write_HERC_profiles()
        hercplanet.write_yaml(output_dir='../output')

DMO_M0.455PyroSChotSMhotN128.txt
Saved file:  ./HERC_profiles/DMO_M0.455PyroSChotSMhot_HERCULES_core_profiles.csv
Saved file:  ./HERC_profiles/DMO_M0.455PyroSChotSMhot_HERCULES_mant_profiles.csv
Saved file:  ./HERC_yaml/DMO_M0.455PyroSChotSMhot_input.yml
DMO_M0.55PyroSChotSMhotN128.txt
Saved file:  ./HERC_profiles/DMO_M0.55PyroSChotSMhot_HERCULES_core_profiles.csv
Saved file:  ./HERC_profiles/DMO_M0.55PyroSChotSMhot_HERCULES_mant_profiles.csv
Saved file:  ./HERC_yaml/DMO_M0.55PyroSChotSMhot_input.yml
DMO_M0.65PyroSChotSMhotN128.txt
Saved file:  ./HERC_profiles/DMO_M0.65PyroSChotSMhot_HERCULES_core_profiles.csv
Saved file:  ./HERC_profiles/DMO_M0.65PyroSChotSMhot_HERCULES_mant_profiles.csv
Saved file:  ./HERC_yaml/DMO_M0.65PyroSChotSMhot_input.yml
DMO_M0.706PyroSChotSMhotN128.txt
Saved file:  ./HERC_profiles/DMO_M0.706PyroSChotSMhot_HERCULES_core_profiles.csv
Saved file:  ./HERC_profiles/DMO_M0.706PyroSChotSMhot_HERCULES_mant_profiles.csv
Saved file:  ./HERC_yaml/DMO_M0.706PyroSChotSMho

Saved file:  ./HERC_profiles/M0.65PyroSCwarmSMwarm_HERCULES_core_profiles.csv
Saved file:  ./HERC_profiles/M0.65PyroSCwarmSMwarm_HERCULES_mant_profiles.csv
Saved file:  ./HERC_yaml/M0.65PyroSCwarmSMwarm_input.yml
M0.706PyroSCwarmSMwarmN128.txt
Saved file:  ./HERC_profiles/M0.706PyroSCwarmSMwarm_HERCULES_core_profiles.csv
Saved file:  ./HERC_profiles/M0.706PyroSCwarmSMwarm_HERCULES_mant_profiles.csv
Saved file:  ./HERC_yaml/M0.706PyroSCwarmSMwarm_input.yml
M0.91PyroSCwarmSMwarmN171_fulltension.txt
Saved file:  ./HERC_profiles/M0.91PyroSCwarmSMwarm_HERCULES_core_profiles.csv
Saved file:  ./HERC_profiles/M0.91PyroSCwarmSMwarm_HERCULES_mant_profiles.csv
Saved file:  ./HERC_yaml/M0.91PyroSCwarmSMwarm_input.yml
M1.3PyroSChotSMwarmN171_fulltension.txt
Saved file:  ./HERC_profiles/M1.3PyroSChotSMwarm_HERCULES_core_profiles.csv
Saved file:  ./HERC_profiles/M1.3PyroSChotSMwarm_HERCULES_mant_profiles.csv
Saved file:  ./HERC_yaml/M1.3PyroSChotSMwarm_input.yml
M1.5PyroSCmeltSMwarmN128.txt
Saved fil

In [5]:
hercstring = '/home/apostema/hercules/hercules/HERCULES_development/Source_code/hercules'

with open('chapter3_hercules.slurm','w') as slurm:
    slurm.write('#!/bin/sh'+'\n')
    slurm.write('#SBATCH -p low #name of partition'+'\n')
    slurm.write('#SBATCH -A publicgrp'+'\n')
    slurm.write('#SBATCH -J HERC_Ch3 # name of run'+'\n')
    slurm.write('#SBATCH -n 32 #number of cores'+'\n')
    slurm.write('#SBATCH -N 1 # use one node for threaded run'+'\n')
    slurm.write('#SBATCH -t 5-00:00:00 # max wall time in days-hours:minutes:seconds format'+'\n')
    slurm.write('#SBATCH --mem=100000 #memory requested per node in mB'+'\n')
    slurm.write('#SBATCH -o stdoutHERCULES.txt'+'\n')
    slurm.write('#SBATCH -e stderrHERCULES.txt'+'\n')
    slurm.write('#SBATCH --mail-user=anpostema@ucdavis.edu #email this user'+'\n')
    slurm.write('#SBATCH --mail-type=END #mail at end of run'+'\n')
    slurm.write('#ulimit -l unlimited'+'\n')
    for filename in os.listdir("./planet_profiles/"):
        if filename.endswith(".txt") and not filename.endswith(".used_params.txt"):
            planet = hercdict[filename]
            slurm.write(hercstring+' -if '+'./'+planet.basename+'_input.yml'+' -vvv 2>&1 | tee '+'../'+planet.basename+'.output.log'+'\n')

In [6]:
from yaml import CLoader as Loader, CDumper as Dumper
fname='./HERC_yaml/M.005PyroSCmantleSMwarm_input.yml'
with open(fname, 'r') as wfile:
    test=yaml.load(wfile,Loader)
test
    test['Initialization']['flag_start']=1
    test['Initialization']['start_file']=snapshot
    test['Initialization']['flag_format_start_file']
#print(test['Initialization']['flag_start'])

IndentationError: unexpected indent (2951110329.py, line 6)

## List of runs
